# MedExplain AI Prototyping

Imports

In [1]:
from haystack_integrations.document_stores.chroma import ChromaDocumentStore
from haystack import Document
import json

Creating the Datastore

In [2]:
# create documents

with open("../utils/data.json", "r") as f:
  data = json.load(f) 

len(data)

49

In [3]:
docs = []

for doc in data:
    name = doc.get("test_name") or doc.get("name", "Unknown Test")
    url = doc.get("source", "none")

    content = f"{name} measures {doc.get('description', 'this parameter')}."

    if "components" in doc:
        for component in doc["components"]:
            comp_name = component.get("name", "Unknown Component")
            comp_desc = component.get("description", "")
            comp_range = component.get("normal_range", "N/A")
            comp_units = component.get("units", "")
            content += f" {comp_name}: {comp_desc} Normal range: {comp_range} {comp_units}."

    else:
        content += f" The normal range is {doc.get('normal_range', 'N/A')} {doc.get('units', '')}."

    content += f" More info: {url}"

    docs.append(
        Document(
            content=content,
            meta={
                "name": name,
                "url": url
            }
        )
    )

len(docs)


49

clean the docs using DocumentPreProcessor

In [4]:
# from haystack.components.preprocessors import DocumentPreprocessor

In [5]:
# preprocessor = DocumentPreprocessor(split_by="word", split_length=150, split_overlap=30)

In [6]:
# cleaned_docs = preprocessor.run(documents=docs)

In [7]:
document_store = ChromaDocumentStore(distance_function='cosine', persist_path="./chroma_documents")
# document_store.write_documents(docs)

In [8]:
document_store.count_documents()

48

Creating a Retriever

In [9]:
from haystack_integrations.components.retrievers.chroma import ChromaQueryTextRetriever

In [10]:
retriever = ChromaQueryTextRetriever(document_store=document_store, top_k=1)

In [11]:
retriever.run(query="TSH W/REFLEX TO FT4 result is 2.65")

{'documents': [Document(id=b13d7d48c97a8d5b7df8f9e8187f43c8b8cea15bfe152eebc29a8f2d8b9034b3, content: 'Thyroid Panel measures A thyroid panel measures the levels of thyroid hormones in your blood to chec...', meta: {'url': 'https://medlineplus.gov/lab-tests/thyroid-function-tests/', 'name': 'Thyroid Panel'}, score: 0.5891539454460144, embedding: vector of size 384)]}

Creating the prompt builder

In [12]:
template = """
<s>[INST]You are a helpful medical assistant.

Patient Lab Results are : {{user_query}}

Context:
{% for doc in documents %}
{{ doc.content }}
{% endfor %}

Instructions:
- Identify the test and what it measures.
- Compare the patient’s result to the normal range.
- Say whether the result is normal or concerning.
- If concerning, briefly mention a possible general health impact.
- Use only 2 short sentences. Do not diagnose.

Now write your response:
[/INST]
"""

In [13]:
from haystack.components.builders.prompt_builder import PromptBuilder

In [14]:
prompt_builder = PromptBuilder(template=template, required_variables=['user_query', 'documents'])

In [15]:
res = prompt_builder.run(
  user_query= "TSH W/REFLEX TO FT4 result is 2.65",
  documents= retriever.run(query="TSH W/REFLEX TO FT4 result is 2.65")
)

In [16]:
res.keys()

dict_keys(['prompt'])

Creating Generator component

In [17]:
from haystack import component
from haystack.dataclasses import Document
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

/Users/tarun/Documents/Projects/medexplain-ai/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
@component
class T5FlanSummarizer:
  """
  A Component generating summaries based on the prompt
  """
  def __init__(self):
    self.tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
    self.model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
    self.model.eval()

  @component.output_types(summary=str)
  def run(self, prompt:str):
    print(prompt)
    inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True)
    with torch.no_grad():
      outputs = self.model.generate(**inputs, max_new_tokens=100)
    summary = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
    return {"summary" : summary}
    

In [19]:
from llama_cpp import Llama
from haystack import component

@component
class MistralGGUFGenerator:
    def __init__(self, model_path: str):
        self.llm = Llama(
            model_path=model_path,
            n_ctx=2048,
            n_threads=4  # Adjust based on your CPU
        )

    @component.output_types(summary=str)
    def run(self, prompt: str):
        output = self.llm(prompt, max_tokens=200, temperature=0.7, stop=["\n"])
        text = output["choices"][0]["text"].strip()
        return {"summary": text}

In [20]:
generator = MistralGGUFGenerator(model_path="../mistral-7b-instruct-v0.1.Q4_K_M.gguf")

llama_model_load_from_file_impl: using device Metal (Apple M2 Pro) - 10922 MiB free
llama_model_loader: loaded meta data with 20 key-value pairs and 291 tensors from ../mistral-7b-instruct-v0.1.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.1
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimensio

Creating the pipeline

In [21]:
from haystack import Pipeline

In [22]:
p = Pipeline()

In [23]:
p.add_component("retriever", retriever)
p.add_component("prompt_builder", prompt_builder)
p.add_component("generator", generator)

In [24]:
p.connect("retriever.documents", "prompt_builder.documents")
p.connect("prompt_builder.prompt", "generator")

🚅 Components
  - retriever: ChromaQueryTextRetriever
  - prompt_builder: PromptBuilder
  - generator: MistralGGUFGenerator
🛤️ Connections
  - retriever.documents -> prompt_builder.documents (List[Document])
  - prompt_builder.prompt -> generator.prompt (str)

In [25]:
p.run({
    "retriever": {"query": "VITAMIN D,25-OH,TOTAL,IA result is  12*"},
    "prompt_builder": {"user_query": "VITAMIN D,25-OH,TOTAL,IA result is  12*"}
})

llama_perf_context_print:        load time =    5611.12 ms
llama_perf_context_print: prompt eval time =    5610.88 ms /   208 tokens (   26.98 ms per token,    37.07 tokens per second)
llama_perf_context_print:        eval time =    6015.53 ms /   105 runs   (   57.29 ms per token,    17.45 tokens per second)
llama_perf_context_print:       total time =   11640.70 ms /   313 tokens


{'generator': {'summary': "The test measures Vitamin D (25-OH) levels in the blood. The patient's result of 12 ng/mL falls outside the normal range of 30 to 50 ng/mL. The result is concerning and may indicate a potential deficiency in vitamin D, which can impact overall health, including bone health and immune function. More information can be found at <https://medlineplus.gov/lab-tests/vitamin-d-test/>"}}

In [26]:
print(p._connection_type_validation)

True
